# 🛡️ Custom Data — Jailbreak Safety Circuits

CircuitKit isn't limited to built-in tasks. You can bring **any CSV** and define prompt/answer templates to discover circuits for your own research question.

This notebook uses a jailbreak safety dataset to discover which circuits distinguish safe from unsafe requests in an instruction-tuned model.

We demonstrate both data paths:
- **Section A:** Paired data (EAP-IG) — clean vs. corrupt prompts
- **Section B:** Clean-only data (IBCircuit) — no corruption needed

- **Model:** `Qwen/Qwen2.5-1.5B-Instruct` (~1.5B params)
- **Dataset:** `jailbreak_binary.csv` (334 rows)
- **Runtime:** ~15-20 min on Colab T4

---

## Setup

In [ ]:
# ⚠️ Must be set before any CUDA context is created.
# If you've already run GPU cells in a prior notebook, restart the runtime first.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

import warnings
warnings.filterwarnings('ignore', module='transformer_lens')
warnings.filterwarnings('ignore', module='lm_eval')
warnings.filterwarnings('ignore', message='.*pretrained.*model kwarg.*')

import gc
import torch
import pandas as pd
from circuitkit import Pipeline

print(f"CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

## Explore the Dataset

The CSV has 5 columns that map to clean/corrupt prompt-answer pairs:

| Column | Role |
|--------|------|
| `system_jailbreak` | System prompt (shared) |
| `benign_req` | Clean user request |
| `harmful_req` | Corrupt user request |
| `clean_ans` | Expected answer to benign request |
| `corrupt_ans` | Expected answer to harmful request |

In [ ]:
# Path relative to the cloned repo root
CSV_PATH = "src/circuitkit/data/task_data/tasks/binary_align/jailbreak_binary.csv"

df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

## Template Syntax

CircuitKit uses Python-style `{column_name}` placeholders in prompt templates. Each row in the CSV fills in the fields to create a concrete prompt.

For this dataset:
```
Clean prompt:   "System: {system_jailbreak}\nUser: {benign_req}\n...\nAssistant:"
Corrupt prompt: "System: {system_jailbreak}\nUser: {harmful_req}\n...\nAssistant:"
Clean answer:   "{clean_ans}"    → e.g. "Yes"
Corrupt answer: "{corrupt_ans}"  → e.g. "No"
```

---
## Section A: Paired Data (EAP-IG)

Paired algorithms compare the model's behavior on clean vs. corrupt inputs to find which circuits are responsible for the behavioral difference.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Define prompt/answer templates
CLEAN_PROMPT = (
    "System: {system_jailbreak}\n"
    "User: {benign_req}\n"
    "Please answer with only 'Yes' or 'No'.\n"
    "Assistant:"
)
CORRUPT_PROMPT = (
    "System: {system_jailbreak}\n"
    "User: {harmful_req}\n"
    "Please answer with only 'Yes' or 'No'.\n"
    "Assistant:"
)
CLEAN_ANSWER = "{clean_ans}"
CORRUPT_ANSWER = "{corrupt_ans}"

In [ ]:
pipe_paired = Pipeline.from_custom_data(
    MODEL,
    data_path=CSV_PATH,
    clean_prompt=CLEAN_PROMPT,
    corrupt_prompt=CORRUPT_PROMPT,
    clean_answer=CLEAN_ANSWER,
    corrupt_answer=CORRUPT_ANSWER,
    task_name="jailbreak_paired",
    output_dir="./results/jailbreak/paired",
)

print(pipe_paired)

In [ ]:
pipe_paired.discover(
    algorithm="eap-ig",
    level="node",
    sparsity=0.3,
    n_examples=256,
    batch_size=2,
    ig_steps=5,
)

print(f"Circuit: {pipe_paired._circuit}")
print(f"\nTop 10 nodes:")
for name, score in pipe_paired._circuit.top_nodes(10).items():
    print(f"  {name:>10s}  {score:.4f}")

### Evaluate the Paired Circuit

Run all 5 evaluation pillars (excluding generalization, which requires a different task).

In [ ]:
pipe_paired.evaluate(
    pillars=["patching", "ablation", "baselines", "robustness", "stability"],
    n_examples=256,
    n_stability_runs=3,
)

if pipe_paired._eval_report:
    import json
    print(json.dumps(pipe_paired._eval_report, indent=2, default=str))

In [ ]:
# Release pipe_paired's model before Section B loads its own.
# pipe_paired._model is set during the custom-data discover() path;
# freeing it here prevents two full models from being resident simultaneously.
pipe_paired._model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## Section B: Clean-Only Data (IBCircuit)

IBCircuit uses an information bottleneck objective and only needs clean inputs — no corruption pairs required. This is useful when meaningful corruptions are hard to define.

In [ ]:
pipe_clean = Pipeline.from_custom_data(
    MODEL,
    data_path=CSV_PATH,
    clean_prompt=CLEAN_PROMPT,
    clean_answer=CLEAN_ANSWER,
    # No corrupt_prompt / corrupt_answer → clean-only path
    task_name="jailbreak_clean_only",
    output_dir="./results/jailbreak/clean_only",
)

pipe_clean.discover(
    algorithm="ibcircuit",
    level="node",
    sparsity=0.3,
    n_examples=256,
    batch_size=2,
    num_epochs=1000,
    learning_rate=0.05,
)

print(f"IBCircuit: {pipe_clean._circuit}")
print(f"\nTop 10 nodes:")
for name, score in pipe_clean._circuit.top_nodes(10).items():
    print(f"  {name:>10s}  {score:.4f}")

In [ ]:
# Release pipe_clean's model before the visualization cell.
pipe_clean._model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Compare Paired vs. Clean-Only Circuits

Do EAP-IG and IBCircuit find similar safety-critical circuits?

In [ ]:
from circuitkit.visualize.comparison import ComparisonDashboard

# Build comparison if both circuits have scores
if pipe_paired._circuit and pipe_paired._circuit.scores and \
   pipe_clean._circuit and pipe_clean._circuit.scores:

    dashboard = ComparisonDashboard(
        circuits={
            "EAP-IG (paired)": pipe_paired._circuit.scores,
            "IBCircuit (clean-only)": pipe_clean._circuit.scores,
        },
        comparison_type="stability",
    )

    fig = dashboard.plot_stability_heatmap(top_k=15)
    fig.show()

    fig = dashboard.plot_correlation_matrix()
    fig.show()
else:
    print("One or both circuits missing scores — cannot compare.")

## Key Takeaways

1. **Template syntax** `{column_name}` maps CSV columns to prompt positions
2. **Paired algorithms** (EAP, EAP-IG, ACDC) need both clean and corrupt templates
3. **Clean-only algorithms** (IBCircuit, CDT) need only the clean template
4. `Pipeline.from_custom_data()` stores the config; actual task registration happens at `.discover()` time
5. The same CSV can power both paired and clean-only experiments — just change the constructor args